In [ ]:
#!python -m spacy download en_core_web_sm

In [46]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import wordcloud

In [17]:
# # modeli za ustrezno obdelavo stavkov, besed, ločil....
# nltk.download('punkt')     # stavki, besede
# nltk.download('wordnet') #lemmatizacija
# nltk.download('averaged_perceptron_tagger') #POS tagganje
# nltk.download('omw-1.4') 
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_percep

True

In [ ]:
# # tokenization and lemmatization 
# lemmatizer= WordNetLemmatizer()

In [ ]:
# # pokupčkamo besede s podobnim korenom, pomenom skupaj
# # run, runs, running -> run
# def get_wordnet_pos(treebank_tag):
#     if treebank_tag.startswith('J'):
#         return wordnet.ADJ
#     elif treebank_tag.startswith('V'):
#         return wordnet.VERB
#     elif treebank_tag.startswith('RB'):
#         return wordnet.ADV
#     elif treebank_tag.startswith('N'):
#         return wordnet.NOUN
#     else:
#         return wordnet.NOUN

In [ ]:
# def tokenize_lematize(tekst):
#     tokens = word_tokenize(tekst.lower())  # vse besede pišemo z malo začetnico
#     tagged = pos_tag(tokens)
#     return [
#         lemmatizer.lemmatize(word, get_wordnet_pos(tag))
#         for word, tag in tagged
#         if word.isalpha()    # znebimo se ločil, st,...
#     ]

In [47]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [48]:
def tokenize_lematize(tekst):
    doc = nlp(tekst)
    
    izbrane_besede = []
    
    # želimo odstraniti osebe, kraji, jeziki, narodi...
    odstrani_pos = ['PROPN', 'PRON', 'VERB', 'ADV']
    odstrani_entitete = {'PERSON', 'GPE', 'LOC', 'NORP', 'FAC', 'ORG'}
    
    #mnozica prepoznanih entitet
    for token in doc:
        # odstranimo i
        if token.pos_ in odstrani_pos:
            continue
        if token.ent_type_ in odstrani_entitete:
            continue
            
        # spacy ima oznake NOUN, ADJ
        if token.pos_ in ['NOUN', 'ADJ']:
            beseda = token.lemma_.lower()
            # samo crke in dolžina nad 2 znaka
            if beseda.isalpha() and len(beseda) > 2:
                izbrane_besede.append(beseda)
                
    return izbrane_besede

In [49]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()


custom_words = {
    # založniški šum 
    'book', 'novel', 'story', 'author', 'literature', 'edition', 'seller', 
    'read', 'reader', 'page', 'chapter', 'write', 'writer', 'publish', 
    'publication', 'review', 'times', 'york', 'print', 'copy', 'best', 
    'original', 'classic', 'introduction', 'note', 'debut', 'thriller',
    'unique', 'fascinating', 'scandal', 'major', 'character', 'cover', 'magazine',
    'self', 'series', 'volume', 'masterpiece', 'translation', 'film', 'tale', 'course',
    
    # splošni opisi 
    'way', 'thing', 'important', 'practical', 'young', 'boy', 'girl', 
    'human', 'people', 'great', 'good', 'bad', 'true', 'new', 'old',
    'life', 'world', 'everything', 'day', 'time', 'year', 'make',
    'take', 'come', 'think', 'feel', 'know', 'look', 'want', 'large', 'small',
    'man', 'woman', 'literary', 'secret', 'isbn', 'mother', 'sister', 'father',
    'little', 'room', 'place', 'end', 'first', 'second',
    
    #iz izpisa
    'professional', 'guide', 'experience', 'natural', 'vivid', 'narrative',
    'compelling', 'extraordinary', 'powerful', 'voice', 'mind'
}


all_stopwords = list(base_stopwords.union(custom_words))

In [50]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)

#random.seed(42)
# vzamemo 150/200 knjig, ostale bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\leto_2003\03_ang_opisi\*.txt')[:150]
#filepaths= random.sample(filepaths, 150)
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
# custom_stopwords = list(text.ENGLISH_STOP_WORDS.union({'book', 'novel', 'story', 'author', 'character',
#                                                   'edition', 'classic', 'bestseller', 'review',
#                                                   'read', 'reader', 'york', 'times', 'new'}))
vectorizer= CountVectorizer(stop_words= all_stopwords, #custom_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=3, 
                            max_df=0.7)

In [51]:
X = vectorizer.fit_transform(filepaths) 


c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [52]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2807 stored elements and shape (150, 551)>
  Coords	Values
  (0, 166)	1
  (0, 128)	2
  (0, 405)	1
  (0, 334)	1
  (0, 350)	1
  (0, 414)	1
  (0, 369)	1
  (0, 509)	1
  (0, 354)	1
  (0, 378)	1
  (0, 213)	1
  (0, 190)	2
  (0, 466)	1
  (0, 105)	1
  (0, 464)	1
  (0, 121)	1
  (0, 200)	1
  (0, 1)	1
  (0, 487)	1
  (0, 120)	1
  (0, 367)	1
  (0, 292)	1
  (1, 128)	1
  (1, 200)	1
  (1, 181)	1
  :	:
  (147, 534)	3
  (147, 207)	1
  (147, 119)	1
  (147, 87)	1
  (147, 67)	3
  (147, 511)	1
  (147, 242)	1
  (147, 335)	1
  (147, 54)	1
  (147, 276)	1
  (147, 330)	1
  (147, 444)	1
  (148, 334)	1
  (148, 350)	1
  (148, 375)	1
  (148, 73)	1
  (149, 354)	1
  (149, 105)	1
  (149, 224)	1
  (149, 326)	1
  (149, 421)	1
  (149, 115)	1
  (149, 550)	2
  (149, 57)	1
  (149, 177)	2
[[0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 2]]
   account  action  actual  addition  adolescence  adventure  

In [53]:
from sklearn.decomposition import PCA

In [57]:
pca = PCA(n_components=4)  
X_pca = pca.fit_transform(X)

In [58]:
print(pca.explained_variance_ratio_)
print("Skupaj:", sum(pca.explained_variance_ratio_))

[0.04439462 0.03336372 0.02672725 0.02540044]
Skupaj: 0.1298860437276807


In [60]:
feature_names = vectorizer.get_feature_names_out()

for i, comp in enumerate(pca.components_):
    top_words = [feature_names[j] for j in comp.argsort()[-10:]]
    print(f"PC{i+1}: {top_words}")

PC1: ['camp', 'city', 'child', 'family', 'empire', 'country', 'historical', 'war', 'history', 'love']
PC2: ['medical', 'wedding', 'marriage', 'city', 'power', 'prison', 'heart', 'child', 'family', 'love']
PC3: ['country', 'wife', 'account', 'house', 'fundamentalist', 'brother', 'violent', 'religious', 'faith', 'family']
PC4: ['savage', 'bone', 'religious', 'violence', 'empire', 'work', 'realm', 'fundamentalist', 'violent', 'faith']


In [29]:
# import matplotlib.pyplot as plt
# labels = X_pca.argmax(axis=1)   

# pairs = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]

# for i, j in pairs:
#     plt.figure()
#     plt.scatter(
#         X_pca[:, i],
#         X_pca[:, j],
#         c=labels,       
#         cmap="tab10"    
#     )
#     plt.xlabel(f"PC{i+1}")
#     plt.ylabel(f"PC{j+1}")
#     plt.title(f"PC{i+1} vs PC{j+1}")
#     plt.colorbar(label="žanr")   
#     plt.show()

# plt.show()